# 01 — Tools, not agents

**Goal.** Before we bring in an LLM, build the *tools* an agent will eventually call. A "tool" is just a Python function plus a JSON schema describing its inputs. Demystifying this is the most important first step in agentic AI.

**Scope.** No model in the loop. No `ollama.chat()`. We'll wrap four functions over the DuckDB substrate from cti_401 and end with a tool *registry* — a dict mapping tool names to (schema, callable) pairs. nb 02 picks this up and lets Llama 3.1 choose which tool to call.

**Inputs.** `../cti_401/db/cti.duckdb` (already populated by cti_401 notebooks 03–06).

## Why "tools, not agents" first

In every agentic system the model does *not* execute code itself. It emits structured text saying "please call function X with arguments Y," your code runs the function, and the result goes back to the model. That's it. Once you've seen the four pieces — schema, function, dispatch, return — the rest of agentic AI is just *what the model decides to do with them*.

## Lightweight design rules (8 GB GPU, local Llama 3.1 8B)

Local 7–8B models have weaker tool-use than frontier models. Three rules keep failures bounded:

1. **Tools return small payloads.** Truncate text to a few hundred chars. Big bodies stay in DuckDB; tools return summaries and IDs.
2. **Schemas are minimal and explicit.** Fewer parameters, simple types, clear descriptions. Optional parameters get sensible defaults.
3. **Tools never raise on bad input.** They return an error dict. The model can read it and adapt; an exception kills the whole turn.

## 1. Setup

The only dependency right now is `duckdb`. We point at cti_401's database in **read-only** mode for the query tools — that way nothing in this notebook can corrupt the corpus you spent five notebooks building.

In [18]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import duckdb
import requests

DB_PATH = Path("../cti_401/db/cti.duckdb")
assert DB_PATH.exists(), f"Run cti_401 first. Not found: {DB_PATH.resolve()}"

# A single read-only connection used by all query tools.
# Read-only is deliberate: tools should not be able to mutate the corpus.
RO_CON = duckdb.connect(str(DB_PATH), read_only=True)

print("connected to:", DB_PATH.resolve())
print("tables:", [r[0] for r in RO_CON.execute("SHOW TABLES").fetchall()])

connected to: /home/saqib/cti/cti_401/db/cti.duckdb
tables: ['alerts', 'documents']


## 2. Tool 1 — `query_documents`

List documents in the corpus, optionally filtered by source or label. Returns a small payload (URL ID, source, label, score, title) — never the full text. The model uses this to *find* documents; it uses tool 2 to *read* one.

Notice the shape:

- A **schema** dict (JSON-Schema-flavoured) describing inputs.
- A **function** that takes those inputs as kwargs and returns a JSON-serialisable result.
- A **default `limit`** so a careless agent can't dump 10 000 rows into the prompt.

In [19]:
QUERY_DOCUMENTS_SCHEMA = {
    "name": "query_documents",
    "description": (
        "List documents from the CTI corpus. Returns a compact list with url_id, "
        "source, label, score, and title. Use this to find documents; use "
        "get_document to read the full text of one."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "source": {
                "type": "string",
                "description": "Filter by source. One of: 'onion', 'ransomware_live'. Omit for all.",
                "enum": ["onion", "ransomware_live"],
            },
            "label": {
                "type": "string",
                "description": "Filter by classifier label (e.g. 'Hacking', 'Crypto', 'Violence'). Omit for all.",
            },
            "limit": {
                "type": "integer",
                "description": "Max rows to return. Default 10, max 25.",
            },
        },
        "required": [],
    },
}

def query_documents(source=None, label=None, limit=10):
    limit = max(1, min(int(limit or 10), 25))
    where, args = [], []
    if source:
        where.append("source = ?")
        args.append(source)
    if label:
        where.append("label = ?")
        args.append(label)
    clause = ("WHERE " + " AND ".join(where)) if where else ""
    rows = RO_CON.execute(f"""
        SELECT url_id, source, label, score, title
        FROM documents
        {clause}
        ORDER BY fetched_at DESC NULLS LAST
        LIMIT {limit}
    """, args).fetchall()
    return {
        "count": len(rows),
        "results": [
            {
                "url_id": r[0],
                "source": r[1],
                "label": r[2],
                "score": round(r[3], 3) if r[3] is not None else None,
                "title": (r[4] or "")[:120],
            }
            for r in rows
        ],
    }

# Smoke test by hand
query_documents(source="onion", limit=5)

{'count': 4,
 'results': [{'url_id': 'f77e59ad053e076d99eecfabd4f2ed043c5c8807',
   'source': 'onion',
   'label': 'Hacking',
   'score': 0.964,
   'title': 'Share and accept documents securely'},
  {'url_id': '44f1755f8ced2a54aef914fb35413e59cc77ecca',
   'source': 'onion',
   'label': 'Crypto',
   'score': 0.891,
   'title': 'Ahmia'},
  {'url_id': '770f1d9e993c747bc59f1484e6eea49d7832d37a',
   'source': 'onion',
   'label': 'Violence',
   'score': 0.984,
   'title': 'BBC - Home'},
  {'url_id': '17fb89e941d731022e62715e2774dcd3c57ddec3',
   'source': 'onion',
   'label': 'Crypto',
   'score': 0.967,
   'title': 'The Tor Project | Privacy & Freedom Online'}]}

## 3. Tool 2 — `get_document`

Fetch the full text of a single document by `url_id`. **Truncated to 1500 characters** because we are running a local 8B model with an 8K context window — long tool outputs are the most common reason agent loops blow up.

Notice the explicit error path: if the `url_id` doesn't exist, the tool returns `{"error": ...}` rather than raising. The model will read that and try a different URL.

In [20]:
GET_DOCUMENT_SCHEMA = {
    "name": "get_document",
    "description": (
        "Return the metadata and a truncated text body for one document. "
        "The full text may be longer than what is returned; the truncated flag indicates this."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url_id": {
                "type": "string",
                "description": "The url_id (sha1) returned by query_documents.",
            },
            "max_chars": {
                "type": "integer",
                "description": "Max characters of text to return. Default 1500, max 3000.",
            },
        },
        "required": ["url_id"],
    },
}

def get_document(url_id, max_chars=1500):
    if not url_id:
        return {"error": "url_id is required"}
    max_chars = max(200, min(int(max_chars or 1500), 3000))
    row = RO_CON.execute("""
        SELECT url_id, url, source, title, label, score, n_chars, fetched_at, text
        FROM documents WHERE url_id = ?
    """, [url_id]).fetchone()
    if row is None:
        return {"error": f"no document with url_id={url_id!r}"}
    text = row[8] or ""
    truncated = len(text) > max_chars
    return {
        "url_id": row[0],
        "url": row[1],
        "source": row[2],
        "title": row[3],
        "label": row[4],
        "score": round(row[5], 3) if row[5] is not None else None,
        "n_chars_total": row[6],
        "fetched_at": str(row[7]),
        "text": text[:max_chars],
        "truncated": truncated,
    }

# Smoke test: pick the first doc returned above and read it
first = query_documents(limit=1)["results"][0]
doc = get_document(first["url_id"])
print("title:", doc["title"])
print("truncated:", doc["truncated"], "| total chars:", doc["n_chars_total"])
print("preview:", doc["text"][:300], "...")

title: Share and accept documents securely
truncated: False | total chars: 790
preview: Latest News
SecureDrop Inbox 1.0.0 releasedShare documents securely with these organizations
The Washington Post
The Washington Post is an American daily newspaper published in Washington, DC.
The Guardian
The Guardian is a British daily newspaper.
Disclose
A French non-profit investigative media or ...


## 4. Tool 3 — `query_alerts`

Read recent alerts from the `alerts` table that nb 06 populates. The agent will use this to answer monitoring questions like "what high-severity alerts came in this week?"

In [21]:
QUERY_ALERTS_SCHEMA = {
    "name": "query_alerts",
    "description": (
        "List recent alerts from the monitoring loop. Returns a compact list with "
        "alert_id, severity, source, label, score, title, and raised_at."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "severity": {
                "type": "string",
                "description": "Filter by severity. One of: 'high', 'medium'. Omit for all.",
                "enum": ["high", "medium"],
            },
            "since_days": {
                "type": "integer",
                "description": "Only alerts raised within this many days. Default 30.",
            },
            "limit": {
                "type": "integer",
                "description": "Max rows to return. Default 10, max 25.",
            },
        },
        "required": [],
    },
}

def query_alerts(severity=None, since_days=30, limit=10):
    limit = max(1, min(int(limit or 10), 25))
    since_days = max(1, int(since_days or 30))
    # since_days is inlined as an int: DuckDB does not accept a parameter
    # placeholder inside `INTERVAL N DAY`. since_days is already clamped to
    # a small positive int above, so this is not a SQL-injection vector.
    where = [f"raised_at >= now() - INTERVAL {since_days} DAY"]
    args = []
    if severity:
        where.append("severity = ?")
        args.append(severity)
    rows = RO_CON.execute(f"""
        SELECT alert_id, severity, source, label, score, title, raised_at
        FROM alerts
        WHERE {' AND '.join(where)}
        ORDER BY raised_at DESC
        LIMIT {limit}
    """, args).fetchall()
    return {
        "count": len(rows),
        "results": [
            {
                "alert_id": r[0],
                "severity": r[1],
                "source": r[2],
                "label": r[3],
                "score": round(r[4], 3) if r[4] is not None else None,
                "title": (r[5] or "")[:120],
                "raised_at": str(r[6]),
            }
            for r in rows
        ],
    }

query_alerts(severity="high", since_days=30)

{'count': 1,
 'results': [{'alert_id': 'f77e59ad053e076d99eecfabd4f2ed043c5c8807',
   'severity': 'high',
   'source': 'onion',
   'label': 'Hacking',
   'score': 0.964,
   'title': 'Share and accept documents securely',
   'raised_at': '2026-04-29 20:43:38.529845'}]}

## 5. Tool 4 — `count_recent_victims`

A read-only check against the public ransomware.live API for how many victims have been posted in the last N days. Counts only — no ingestion, no DB writes. The agent uses this to answer "is there anything new since yesterday?"

Note: this is the only tool that does **network I/O**. The others hit DuckDB locally. We add a hard timeout so a stalled API call can't hang the agent loop.

In [22]:
COUNT_RECENT_VICTIMS_SCHEMA = {
    "name": "count_recent_victims",
    "description": (
        "Query the public ransomware.live API for the count of recent ransomware victims. "
        "Read-only; does not modify the corpus. Returns total fetched and how many fall "
        "within the requested lookback window."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "lookback_days": {
                "type": "integer",
                "description": "How many days back to count. Default 7, max 90.",
            },
        },
        "required": [],
    },
}

def count_recent_victims(lookback_days=7):
    lookback_days = max(1, min(int(lookback_days or 7), 90))
    try:
        r = requests.get(
            "https://api.ransomware.live/v2/recentvictims",
            headers={"User-Agent": "cti_501 agent (research)"},
            timeout=15,
        )
        r.raise_for_status()
        victims = r.json()
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

    since = datetime.now(timezone.utc) - timedelta(days=lookback_days)
    in_window = 0
    for v in victims:
        s = v.get("discovered") or v.get("attackdate") or ""
        try:
            dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
        except (ValueError, AttributeError):
            continue
        if dt >= since:
            in_window += 1

    return {
        "total_returned_by_api": len(victims),
        "in_lookback_window": in_window,
        "lookback_days": lookback_days,
    }

count_recent_victims(lookback_days=7)

{'total_returned_by_api': 100, 'in_lookback_window': 0, 'lookback_days': 7}

## 6. The tool registry

Two structures the LLM in nb 02 will consume:

1. **`TOOL_SCHEMAS`** — the list of schemas in the format Ollama's `chat()` expects. Sent to the model as the `tools=` argument.
2. **`TOOL_FUNCTIONS`** — a name → callable map. When the model emits a tool call, our dispatcher looks up the name here and runs the function.

This split is the heart of agentic AI: **the model sees schemas, your code holds the functions.** The model never executes anything; it only requests.

In [23]:
# Ollama expects each tool wrapped as {"type": "function", "function": {...schema...}}
TOOL_SCHEMAS = [
    {"type": "function", "function": QUERY_DOCUMENTS_SCHEMA},
    {"type": "function", "function": GET_DOCUMENT_SCHEMA},
    {"type": "function", "function": QUERY_ALERTS_SCHEMA},
    {"type": "function", "function": COUNT_RECENT_VICTIMS_SCHEMA},
]

TOOL_FUNCTIONS = {
    "query_documents": query_documents,
    "get_document": get_document,
    "query_alerts": query_alerts,
    "count_recent_victims": count_recent_victims,
}

def dispatch_tool_call(name, arguments):
    """Run a tool by name, never raising. Returns the tool's result dict
    or an {'error': ...} dict the model can read and respond to."""
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return {"error": f"unknown tool: {name!r}. "
                          f"Valid tools: {list(TOOL_FUNCTIONS)}"}
    args = arguments if isinstance(arguments, dict) else {}
    try:
        return fn(**args)
    except TypeError as e:
        return {"error": f"bad arguments for {name}: {e}"}
    except ValueError as e:
        return {"error": f"bad argument value for {name}: {e}. "
                          f"Check argument types in the schema."}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

print(f"{len(TOOL_SCHEMAS)} tools registered:")
for t in TOOL_SCHEMAS:
    print(f"  - {t['function']['name']}")

4 tools registered:
  - query_documents
  - get_document
  - query_alerts
  - count_recent_victims


## 7. Smoke test the dispatcher

Pretend we are the LLM. We hand-write tool calls and run them through `dispatch_tool_call`. If this works end-to-end, nb 02 is ready to plug Llama 3.1 in as the *thing that picks which call to make*.

In [24]:
import json

fake_calls = [
    ("query_documents", {"source": "onion", "limit": 3}),
    ("query_documents", {"label": "Hacking"}),
    ("query_alerts", {"severity": "high", "since_days": 30}),
    ("count_recent_victims", {"lookback_days": 30}),
    ("get_document", {"url_id": "does-not-exist"}),                # error path
    ("unknown_tool", {}),                                          # error path
    ("query_documents", {"limit": "not-a-number"}),                # bad args
]

for name, args in fake_calls:
    result = dispatch_tool_call(name, args)
    summary = json.dumps(result)[:200]
    print(f"-> {name}({args})\n   {summary}\n")

-> query_documents({'source': 'onion', 'limit': 3})
   {"count": 3, "results": [{"url_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents securely"}, {"url_id": "44f1

-> query_documents({'label': 'Hacking'})
   {"count": 1, "results": [{"url_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents securely"}]}

-> query_alerts({'severity': 'high', 'since_days': 30})
   {"count": 1, "results": [{"alert_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "severity": "high", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents secure

-> count_recent_victims({'lookback_days': 30})
   {"total_returned_by_api": 100, "in_lookback_window": 0, "lookback_days": 30}

-> get_document({'url_id': 'does-not-exist'})
   {"error": "no document with url_id='does-not-exist'"}

-> unknown_tool({})


## What's next

Notebook 02 imports `TOOL_SCHEMAS` and `TOOL_FUNCTIONS` from this notebook (we'll move them into a small `tools.py` module so they can be `import`ed cleanly), connects to Ollama, and lets Llama 3.1 8B answer questions by *choosing one of these tools*. No planning loop yet — single-step tool use only.

## Recap

An agent is **a loop** around two things you already have:

- **Schemas** the model reads → so it knows what calls are available.
- **Functions** your code runs → so the calls actually do something.

Everything else in agentic AI — planning, memory, multi-agent, guardrails — is built on top of this dispatch primitive. Get this layer right and the rest follows.